# LLM adjudication

Scores TermMap's LLM requests with a self-hosted open-weight model on a GPU runtime.
Runtime > Change runtime type > **T4 GPU**, then Runtime > Run all.

All files are read from and written to one Google Drive folder, `My Drive/HiLabs_Clinical`.

**Inputs** (place in the folder root before running): one or more `llm_requests*.jsonl` files,
written by `python -m termmap evaluate` or `python -m termmap export-llm`.

**Outputs:** `outputs/llm_scores<suffix>.jsonl` for each request file. Copy into the repository's `artifacts/`.
A request file whose scores already exist in `outputs/` is skipped.

The model reads each prompt once and the probability of each candidate letter is taken from the
next-token distribution. There is no sampling, so results are deterministic for a fixed model and input.

In [ ]:
!pip -q install -U transformers accelerate bitsandbytes

In [ ]:
import glob, json, os, time, torch
from google.colab import drive
drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/HiLabs_Clinical"
assert os.path.isdir(WORK), f"create the folder {WORK} in Google Drive first"
CACHE, OUT = f"{WORK}/cache", f"{WORK}/outputs"
os.makedirs(CACHE, exist_ok=True)
os.makedirs(OUT, exist_ok=True)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

assert torch.cuda.is_available(), "switch the runtime to a GPU"
torch.manual_seed(0)
MODEL = "Qwen/Qwen2.5-7B-Instruct"
LETTERS = "ABCDEFGHIJ"
BATCH = 8

def out_path(req_path):
    return f"{OUT}/" + os.path.basename(req_path).replace("llm_requests", "llm_scores")

def ids(p):
    return {json.loads(l)["id"] for l in open(p, encoding="utf-8") if l.strip()}

pending = []
for p in sorted(glob.glob(f"{WORK}/llm_requests*.jsonl")):
    o = out_path(p)
    if os.path.exists(o) and ids(o) == ids(p):
        print("already scored:", os.path.basename(p))
    else:
        pending.append(p)
print("to score:", [os.path.basename(p) for p in pending])

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL)
tok.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(
    MODEL, device_map="auto",
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                                           bnb_4bit_quant_type="nf4"),
).eval()
letter_ids = [tok.encode(L, add_special_tokens=False)[0] for L in LETTERS]
assert len(set(letter_ids)) == len(LETTERS)

In [ ]:
def score(reqs):
    prompts = [tok.apply_chat_template(r["messages"], tokenize=False, add_generation_prompt=True) for r in reqs]
    order = sorted(range(len(reqs)), key=lambda i: len(prompts[i]))  # less padding
    out = [None] * len(reqs)
    t = time.time()
    for b in range(0, len(order), BATCH):
        idx = order[b:b + BATCH]
        enc = tok([prompts[i] for i in idx], return_tensors="pt", padding=True).to(model.device)
        with torch.no_grad():
            logits = model(**enc).logits[:, -1, :].float()
        for i, row in zip(idx, logits):
            n = len(reqs[i]["codes"])
            p = torch.softmax(row[letter_ids[:n]], dim=-1).cpu().tolist()
            out[i] = {"id": reqs[i]["id"], "probs": [round(x, 6) for x in p]}
        if (b // BATCH) % 50 == 0:
            print(f"{b + len(idx)}/{len(reqs)}  {time.time() - t:.0f}s", flush=True)
    return out

for p in pending:
    reqs = [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]
    scores = score(reqs)
    with open(out_path(p), "w", encoding="utf-8") as fh:
        for s in scores:
            fh.write(json.dumps(s) + "\n")
    print(os.path.basename(p), "->", out_path(p), len(scores))